# Ir Além — Opção 2: Otimização de Recursos com Algoritmos Genéticos
### FIAP · Fase 7 · IA Como Fertilizante Digital
**Aluno:** Luan Gonçalves Gomes — RM 566806

---

## Objetivo

Adaptar um **Algoritmo Genético (GA)** para resolver o problema de alocação eficiente de insumos agrícolas na fazenda FarmTech, modelado como uma **mochila binária** (*binary knapsack*).

Além disso, implementar um **Meta-GA** — um segundo algoritmo genético cuja população são *configurações de hiperparâmetros* do GA interno. A analogia com Random Forest é precisa: assim como a floresta executa muitas árvores com variações aleatórias e agrega o melhor resultado, o Meta-GA executa muitas instâncias do GA com configurações distintas e evolui em direção à configuração ótima.

```
Meta-GA (nível 2)
  └─ evolui populações de configurações: [pop_size, mut_rate, cross_rate, selection]
        └─ GA Interno (nível 1) — cada configuração resolve o knapsack agrícola
              └─ Avaliação: qualidade da solução + velocidade de convergência
```

## Dados
Usamos o dataset `crop_yield.csv` da Fase 5 + dados de insumos gerados com semente fixa (reprodutibilidade garantida).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import json
import random
from copy import deepcopy
from dataclasses import dataclass, field
from typing import Callable

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print('Bibliotecas carregadas. Seed:', SEED)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import json
import random
from pathlib import Path
from copy import deepcopy
from dataclasses import dataclass, field
from typing import Callable

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print('Bibliotecas carregadas. Seed:', SEED)

## 1. Geração e Salvamento da Base de Dados (Reprodutibilidade)

Geramos os insumos agrícolas com semente fixa e salvamos em CSV — garantindo que qualquer execução futura produza exatamente os mesmos dados de entrada.

In [ ]:
def gerar_e_salvar_dados(n_insumos: int = 30, budget: float = 50_000.0,
                         seed: int = SEED) -> pd.DataFrame:
    """Gera insumos agrícolas e salva em CSV para reprodutibilidade."""
    rng = np.random.RandomState(seed)

    categorias = ['Fertilizante', 'Defensivo', 'Semente', 'Irrigação', 'Corretivo']
    culturas   = ['Soja', 'Milho', 'Café', 'Cana', 'Trigo']

    df = pd.DataFrame({
        'insumo_id':   range(n_insumos),
        'nome':        [f"{random.choice(categorias)}-{i:02d}" for i in range(n_insumos)],
        'cultura':     [random.choice(culturas) for _ in range(n_insumos)],
        # custo em R$ (peso da mochila)
        'custo_reais': rng.uniform(500, 8_000, n_insumos).round(2),
        # ganho esperado de produtividade (valor da mochila)
        'ganho_ton_ha': rng.uniform(0.5, 15.0, n_insumos).round(3),
        # relação valor/peso (eficiência)
    })
    df['eficiencia'] = (df['ganho_ton_ha'] / df['custo_reais'] * 1000).round(4)

    path = DATA_DIR / 'insumos_farmtech.csv'
    df.to_csv(path, index=False)
    print(f"Dados salvos em '{path}' — {len(df)} insumos | Budget: R$ {budget:,.0f}")
    return df


def carregar_dados() -> tuple[pd.DataFrame, float]:
    """Carrega os dados do CSV salvo."""
    path = DATA_DIR / 'insumos_farmtech.csv'
    if not path.exists():
        raise FileNotFoundError(f"Execute gerar_e_salvar_dados() primeiro.")
    df = pd.read_csv(path)
    budget = 50_000.0
    print(f"Dados carregados de '{path}' — {len(df)} insumos")
    return df, budget


# Gera e salva
df_insumos = gerar_e_salvar_dados(n_insumos=30)

# Relê do arquivo (demonstra reprodutibilidade)
df_insumos, BUDGET = carregar_dados()
df_insumos.head(8)

## 2. Definição do Problema — Mochila Binária Agrícola

**Objetivo:** selecionar quais insumos aplicar para **maximizar o ganho total de produtividade** respeitando o orçamento disponível.

- Cromossomo: vetor binário de tamanho N (1 = aplicar insumo, 0 = não aplicar)
- Fitness: soma de `ganho_ton_ha` dos insumos selecionados, penalizado se o custo exceder o budget

In [ ]:
# Extrai arrays para cálculo vetorizado
CUSTOS  = df_insumos['custo_reais'].values
GANHOS  = df_insumos['ganho_ton_ha'].values
N_GENES = len(CUSTOS)


def fitness(cromossomo: np.ndarray) -> float:
    """Fitness = ganho total; 0 se exceder o budget."""
    custo_total = np.dot(cromossomo, CUSTOS)
    if custo_total > BUDGET:
        return 0.0
    return float(np.dot(cromossomo, GANHOS))


def ganho_e_custo(cromossomo: np.ndarray) -> tuple[float, float]:
    return float(np.dot(cromossomo, GANHOS)), float(np.dot(cromossomo, CUSTOS))


print(f'N_GENES={N_GENES} | Budget=R${BUDGET:,.0f}')
print(f'Ganho máximo possível (todos os itens): {GANHOS.sum():.2f} ton/ha')
print(f'Custo total (todos os itens): R$ {CUSTOS.sum():,.0f}')

## 3. Algoritmo Genético Base (GA Interno)

Implementamos três estratégias intercambiáveis para cada operador genético:

| Operador | Estratégias disponíveis |
|----------|------------------------|
| **Seleção** | Torneio, Roleta, Ranking |
| **Cruzamento** | Ponto único, Dois pontos, Uniforme |
| **Mutação** | Bit-flip simples, Bit-flip adaptativo, Inversão de segmento |

In [ ]:
@dataclass
class ConfigGA:
    """Hiperparâmetros do GA — o que o Meta-GA evolui."""
    pop_size:        int   = 50
    n_geracoes:      int   = 100
    mutation_rate:   float = 0.02
    crossover_rate:  float = 0.80
    elitism:         int   = 2
    selection:       str   = 'tournament'   # tournament | roulette | ranking
    crossover:       str   = 'two_point'    # single | two_point | uniform
    mutation:        str   = 'bitflip'      # bitflip | adaptive | inversion
    tournament_size: int   = 3


# ── Operadores de Seleção ─────────────────────────────────────────────────────

def selecao_torneio(pop, fits, k=3):
    idxs = np.random.choice(len(pop), k, replace=False)
    return pop[idxs[np.argmax(fits[idxs])]].copy()


def selecao_roleta(pop, fits):
    total = fits.sum()
    if total == 0:
        return pop[np.random.randint(len(pop))].copy()
    probs = fits / total
    idx = np.random.choice(len(pop), p=probs)
    return pop[idx].copy()


def selecao_ranking(pop, fits):
    ranks = np.argsort(np.argsort(fits)) + 1
    probs = ranks / ranks.sum()
    idx = np.random.choice(len(pop), p=probs)
    return pop[idx].copy()


# ── Operadores de Cruzamento ──────────────────────────────────────────────────

def crossover_ponto_unico(p1, p2):
    pt = np.random.randint(1, len(p1))
    return np.concatenate([p1[:pt], p2[pt:]]), np.concatenate([p2[:pt], p1[pt:]])


def crossover_dois_pontos(p1, p2):
    a, b = sorted(np.random.choice(len(p1)-1, 2, replace=False) + 1)
    c1 = np.concatenate([p1[:a], p2[a:b], p1[b:]])
    c2 = np.concatenate([p2[:a], p1[a:b], p2[b:]])
    return c1, c2


def crossover_uniforme(p1, p2):
    mask = np.random.randint(0, 2, len(p1), dtype=np.int8)
    return np.where(mask, p1, p2), np.where(mask, p2, p1)


# ── Operadores de Mutação ─────────────────────────────────────────────────────

def mutacao_bitflip(cromossomo, rate):
    mask = np.random.random(len(cromossomo)) < rate
    return np.where(mask, 1 - cromossomo, cromossomo)


def mutacao_adaptativa(cromossomo, rate, geracao, max_ger):
    """Taxa de mutação decai com as gerações (exploração → exploração local)."""
    rate_adj = rate * (1 - geracao / max_ger * 0.7)
    return mutacao_bitflip(cromossomo, rate_adj)


def mutacao_inversao(cromossomo, rate):
    """Inverte um segmento aleatório do cromossomo."""
    if np.random.random() < rate * len(cromossomo):
        a, b = sorted(np.random.randint(0, len(cromossomo), 2))
        cromossomo = cromossomo.copy()
        cromossomo[a:b+1] = cromossomo[a:b+1][::-1]
    return cromossomo


print('Operadores genéticos definidos.')

In [ ]:
def rodar_ga(cfg: ConfigGA, fitness_fn: Callable,
             n_genes: int, verbose: bool = False) -> dict:
    """
    GA principal. Retorna dict com melhor solução, histórico de fitness e tempo.
    """
    t0 = time.perf_counter()

    # Inicialização
    pop  = np.random.randint(0, 2, (cfg.pop_size, n_genes), dtype=np.int8)
    fits = np.array([fitness_fn(ind) for ind in pop])
    historico = [fits.max()]

    # Mapas de operadores
    sel_fn  = {'tournament': lambda p,f: selecao_torneio(p, f, cfg.tournament_size),
               'roulette':   selecao_roleta,
               'ranking':    selecao_ranking}[cfg.selection]

    cx_fn   = {'single':    crossover_ponto_unico,
               'two_point': crossover_dois_pontos,
               'uniform':   crossover_uniforme}[cfg.crossover]

    mut_fn  = {'bitflip':   lambda c, g: mutacao_bitflip(c, cfg.mutation_rate),
               'adaptive':  lambda c, g: mutacao_adaptativa(c, cfg.mutation_rate,
                                                            g, cfg.n_geracoes),
               'inversion': lambda c, g: mutacao_inversao(c, cfg.mutation_rate)
               }[cfg.mutation]

    for ger in range(cfg.n_geracoes):
        nova_pop = []

        # Elitismo
        elite_idxs = np.argsort(fits)[-cfg.elitism:]
        for idx in elite_idxs:
            nova_pop.append(pop[idx].copy())

        # Reprodução
        while len(nova_pop) < cfg.pop_size:
            p1 = sel_fn(pop, fits)
            p2 = sel_fn(pop, fits)
            if np.random.random() < cfg.crossover_rate:
                c1, c2 = cx_fn(p1, p2)
            else:
                c1, c2 = p1.copy(), p2.copy()
            nova_pop.append(mut_fn(c1, ger))
            if len(nova_pop) < cfg.pop_size:
                nova_pop.append(mut_fn(c2, ger))

        pop  = np.array(nova_pop[:cfg.pop_size])
        fits = np.array([fitness_fn(ind) for ind in pop])
        historico.append(fits.max())

    melhor_idx = np.argmax(fits)
    tempo = time.perf_counter() - t0

    return {
        'melhor': pop[melhor_idx],
        'fitness': fits[melhor_idx],
        'historico': historico,
        'tempo_s': round(tempo, 4),
    }


print('Função rodar_ga() definida.')

## 4. GA Base — Execução com Configuração Padrão

Executa o GA com os hiperparâmetros padrão (baseline da aula) para termos uma referência de comparação.

In [ ]:
np.random.seed(SEED)
cfg_baseline = ConfigGA()  # parâmetros padrão
res_baseline = rodar_ga(cfg_baseline, fitness, N_GENES)

ganho_b, custo_b = ganho_e_custo(res_baseline['melhor'])
print(f"\n=== BASELINE ===")
print(f"Fitness  : {res_baseline['fitness']:.4f} ton/ha")
print(f"Custo    : R$ {custo_b:,.2f} / R$ {BUDGET:,.0f}")
print(f"Insumos  : {res_baseline['melhor'].sum()} de {N_GENES}")
print(f"Tempo    : {res_baseline['tempo_s']}s")

## 5. Meta-GA — Otimizando os Hiperparâmetros do GA

### Conceito

O Meta-GA trata cada **configuração de hiperparâmetros** como um indivíduo. Sua população é um conjunto de `ConfigGA` com valores distintos. O fitness de cada indivíduo é calculado executando o GA interno com aquela configuração e avaliando a qualidade da solução encontrada.

```
Geração Meta (t)
  ├── ConfigGA #1: pop=60, mut=0.01, cx=two_point, sel=tournament → fitness_inner=142.3
  ├── ConfigGA #2: pop=30, mut=0.05, cx=uniform,   sel=roulette   → fitness_inner=138.7
  ├── ConfigGA #3: pop=80, mut=0.02, cx=single,    sel=ranking    → fitness_inner=145.1  ← melhor
  └── ...
       ↓  cruzamento e mutação de ConfigGAs
Geração Meta (t+1): novas configurações derivadas das melhores
```

### Espaço de busca dos hiperparâmetros

In [ ]:
# Espaço de busca — cada hiperparâmetro tem faixa discreta ou contínua
HIPER_ESPACO = {
    'pop_size':        [20, 30, 50, 80, 100, 150],
    'mutation_rate':   [0.005, 0.01, 0.02, 0.05, 0.10, 0.15],
    'crossover_rate':  [0.60, 0.70, 0.80, 0.90, 0.95],
    'elitism':         [1, 2, 3, 5],
    'selection':       ['tournament', 'roulette', 'ranking'],
    'crossover':       ['single', 'two_point', 'uniform'],
    'mutation':        ['bitflip', 'adaptive', 'inversion'],
    'tournament_size': [2, 3, 5, 7],
}

# Cromossomo do Meta-GA = índices dentro de cada lista acima
N_META_GENES = len(HIPER_ESPACO)
HIPER_KEYS   = list(HIPER_ESPACO.keys())
HIPER_LENS   = [len(v) for v in HIPER_ESPACO.values()]

print('Espaço de busca:')
for k, v in HIPER_ESPACO.items():
    print(f'  {k:20s}: {v}')


def decodificar_config(cromossomo_meta: np.ndarray) -> ConfigGA:
    """Converte um cromossomo do Meta-GA numa ConfigGA."""
    vals = {k: list(HIPER_ESPACO[k])[int(cromossomo_meta[i]) % len(HIPER_ESPACO[k])]
            for i, k in enumerate(HIPER_KEYS)}
    return ConfigGA(**vals, n_geracoes=60)  # GA interno sempre com 60 gerações


def fitness_meta(cromossomo_meta: np.ndarray) -> float:
    """
    Fitness do Meta-GA: executa o GA interno com a configuração codificada
    e retorna o melhor fitness encontrado, normalizado pelo tempo.
    """
    cfg = decodificar_config(cromossomo_meta)
    res = rodar_ga(cfg, fitness, N_GENES)
    # Penaliza levemente pelo tempo (incentiva soluções rápidas e boas)
    score = res['fitness'] * (1 - 0.05 * min(res['tempo_s'], 5.0) / 5.0)
    return score


print(f'\nN_META_GENES = {N_META_GENES}')

In [ ]:
def rodar_meta_ga(pop_meta: int = 12, n_ger_meta: int = 8,
                  mut_meta: float = 0.25, cx_meta: float = 0.7) -> dict:
    """
    Meta-GA: evolui populações de ConfigGA para encontrar a melhor
    configuração do GA interno.

    Cromossomo: vetor inteiro de N_META_GENES elementos (índices no espaço de busca).
    """
    np.random.seed(SEED)
    t0 = time.perf_counter()

    # Inicialização: cada indivíduo é um vetor de índices aleatórios
    pop = np.array([
        [np.random.randint(0, l) for l in HIPER_LENS]
        for _ in range(pop_meta)
    ])

    print(f"Meta-GA: {pop_meta} configurações × {n_ger_meta} gerações")
    print(f"Total de execuções do GA interno: {pop_meta * n_ger_meta}")
    print("-" * 55)

    historico_meta = []
    melhor_global  = None
    melhor_fit     = -np.inf

    for ger in range(n_ger_meta):
        t_ger = time.perf_counter()

        # Avalia toda a população (executa GA interno para cada configuração)
        fits = np.array([fitness_meta(ind) for ind in pop])

        idx_melhor = np.argmax(fits)
        historico_meta.append(fits.max())

        if fits[idx_melhor] > melhor_fit:
            melhor_fit    = fits[idx_melhor]
            melhor_global = pop[idx_melhor].copy()

        print(f"[Meta-Ger {ger+1:2d}/{n_ger_meta}] "
              f"Melhor={fits.max():.3f}  Média={fits.mean():.3f}  "
              f"Tempo={time.perf_counter()-t_ger:.1f}s")

        # Geração seguinte — elitismo + torneio + cruzamento uniforme + mutação
        nova_pop = [pop[idx_melhor].copy()]  # elitismo = 1

        while len(nova_pop) < pop_meta:
            # Torneio de tamanho 3
            def torneio():
                idxs = np.random.choice(pop_meta, 3, replace=False)
                return pop[idxs[np.argmax(fits[idxs])]].copy()

            p1, p2 = torneio(), torneio()

            # Cruzamento uniforme sobre índices inteiros
            if np.random.random() < cx_meta:
                mask = np.random.randint(0, 2, N_META_GENES).astype(bool)
                filho = np.where(mask, p1, p2)
            else:
                filho = p1.copy()

            # Mutação: troca aleatória de índice em genes selecionados
            for g in range(N_META_GENES):
                if np.random.random() < mut_meta:
                    filho[g] = np.random.randint(0, HIPER_LENS[g])

            nova_pop.append(filho)

        pop = np.array(nova_pop[:pop_meta])

    tempo_total = time.perf_counter() - t0
    melhor_cfg  = decodificar_config(melhor_global)

    print(f"\nMeta-GA concluído em {tempo_total:.1f}s")
    return {
        'melhor_config': melhor_cfg,
        'meta_fitness':  melhor_fit,
        'historico':     historico_meta,
        'tempo_total_s': round(tempo_total, 2),
    }


print("Função rodar_meta_ga() definida.")

In [ ]:
# Executa o Meta-GA
res_meta = rodar_meta_ga(pop_meta=12, n_ger_meta=8)
cfg_otimizada = res_meta['melhor_config']

print("\n=== MELHOR CONFIGURAÇÃO ENCONTRADA PELO META-GA ===")
for k in HIPER_KEYS:
    print(f"  {k:20s}: {getattr(cfg_otimizada, k)}")

## 6. Comparação: Baseline vs. Configuração Otimizada pelo Meta-GA

In [ ]:
# Executa 5 rodadas de cada para comparação estatística
N_RODADAS = 5

resultados_baseline = []
resultados_meta     = []

for i in range(N_RODADAS):
    np.random.seed(i * 7)
    rb = rodar_ga(cfg_baseline,  fitness, N_GENES)
    rm = rodar_ga(cfg_otimizada, fitness, N_GENES)
    resultados_baseline.append(rb)
    resultados_meta.append(rm)

fits_b = [r['fitness'] for r in resultados_baseline]
fits_m = [r['fitness'] for r in resultados_meta]
temps_b = [r['tempo_s'] for r in resultados_baseline]
temps_m = [r['tempo_s'] for r in resultados_meta]

print(f"{'Métrica':30s} {'Baseline':>12s} {'Meta-GA':>12s}")
print("-" * 56)
print(f"{'Fitness médio (ton/ha)':30s} {np.mean(fits_b):>12.3f} {np.mean(fits_m):>12.3f}")
print(f"{'Fitness máximo':30s} {np.max(fits_b):>12.3f} {np.max(fits_m):>12.3f}")
print(f"{'Fitness mínimo':30s} {np.min(fits_b):>12.3f} {np.min(fits_m):>12.3f}")
print(f"{'Desvio padrão':30s} {np.std(fits_b):>12.3f} {np.std(fits_m):>12.3f}")
print(f"{'Tempo médio (s)':30s} {np.mean(temps_b):>12.3f} {np.mean(temps_m):>12.3f}")
delta = (np.mean(fits_m) - np.mean(fits_b)) / np.mean(fits_b) * 100
print(f"\nMelhoria de fitness: {delta:+.1f}%")

## 7. Visualizações

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── Plot 1: Convergência do GA baseline ──────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
for r in resultados_baseline:
    ax1.plot(r['historico'], alpha=0.4, color='steelblue')
ax1.plot(resultados_baseline[0]['historico'], color='steelblue', linewidth=2, label='Baseline')
ax1.set_title('Convergência — Baseline')
ax1.set_xlabel('Geração'); ax1.set_ylabel('Fitness (ton/ha)')
ax1.legend(); ax1.grid(alpha=0.3)

# ── Plot 2: Convergência do GA com config otimizada ───────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
for r in resultados_meta:
    ax2.plot(r['historico'], alpha=0.4, color='darkorange')
ax2.plot(resultados_meta[0]['historico'], color='darkorange', linewidth=2, label='Meta-GA')
ax2.set_title('Convergência — Config Meta-GA')
ax2.set_xlabel('Geração'); ax2.set_ylabel('Fitness (ton/ha)')
ax2.legend(); ax2.grid(alpha=0.3)

# ── Plot 3: Convergência do Meta-GA externo ───────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(res_meta['historico'], marker='o', color='green', linewidth=2)
ax3.set_title('Convergência do Meta-GA\n(evolução de configurações)')
ax3.set_xlabel('Geração Meta'); ax3.set_ylabel('Melhor fitness do GA interno')
ax3.grid(alpha=0.3)

# ── Plot 4: Boxplot comparativo ───────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.boxplot([fits_b, fits_m], labels=['Baseline', 'Meta-GA'],
            patch_artist=True,
            boxprops=dict(facecolor='lightblue'),
            medianprops=dict(color='red', linewidth=2))
ax4.set_title('Distribuição de Fitness (5 rodadas)')
ax4.set_ylabel('Fitness (ton/ha)'); ax4.grid(alpha=0.3, axis='y')

# ── Plot 5: Tempo de execução ─────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
ax5.bar(['Baseline', 'Meta-GA'], [np.mean(temps_b), np.mean(temps_m)],
        color=['steelblue', 'darkorange'], edgecolor='black')
ax5.set_title('Tempo Médio de Execução (s)')
ax5.set_ylabel('Segundos'); ax5.grid(alpha=0.3, axis='y')

# ── Plot 6: Insumos selecionados pela melhor solução ──────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
np.random.seed(SEED)
best_meta = rodar_ga(cfg_otimizada, fitness, N_GENES)
sel_idx = np.where(best_meta['melhor'] == 1)[0]
ax6.barh(df_insumos.iloc[sel_idx]['nome'],
         df_insumos.iloc[sel_idx]['ganho_ton_ha'],
         color='green', alpha=0.7)
ax6.set_title(f'Insumos Selecionados ({len(sel_idx)} de {N_GENES})')
ax6.set_xlabel('Ganho (ton/ha)'); ax6.grid(alpha=0.3, axis='x')

fig.suptitle('FarmTech — Otimização de Insumos com GA e Meta-GA', fontsize=14, fontweight='bold')
plt.savefig(DATA_DIR / 'comparativo_ga_meta_ga.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura salva em data/comparativo_ga_meta_ga.png')

## 8. Conclusão e Justificativa Técnica

### O que foi implementado

| Componente | Descrição |
|------------|-----------|
| **Problema** | Mochila binária agrícola — seleção de insumos FarmTech maximizando produtividade |
| **GA Baseline** | Configuração padrão (aula): torneio, cruzamento 2-pontos, bit-flip |
| **Variações de operadores** | 3 estratégias de seleção × 3 de cruzamento × 3 de mutação |
| **Meta-GA** | GA de nível 2 que evolui configurações do GA interno |
| **Reprodutibilidade** | Dados gerados com semente fixa e salvos em `data/insumos_farmtech.csv` |

### Sobre o Meta-GA

A ideia central é que **não existe uma configuração universal ótima** para um GA — a melhor escolha depende da estrutura do problema. O Meta-GA explora esse espaço de configurações de forma automática, assim como o Random Forest explora o espaço de decisões com múltiplas árvores.

A diferença para a Random Forest é que aqui as "árvores" (configurações de GA) *competem e se reproduzem* entre si, em vez de apenas votarem. Isso permite encontrar combinações sinérgicas de hiperparâmetros que buscas em grade (*grid search*) perderiam.

### Alterações em relação ao baseline

| Ponto de alteração | Baseline | Este trabalho |
|--------------------|----------|---------------|
| `selection()` | Torneio fixo | Torneio / Roleta / Ranking |
| `crossover()` | 2-pontos fixo | Single / 2-pontos / Uniforme |
| `mutation()` | Bit-flip fixo | Bit-flip / Adaptativo / Inversão |
| Hiperparâmetros | Manuais | Otimizados pelo Meta-GA |
| Dados de entrada | Gerados na hora | Salvos em CSV e relidos |

### Limitações

- O Meta-GA aumenta o tempo de computação quadraticamente (cada geração meta executa `pop_meta` GAs internos completos)
- Para problemas muito grandes, técnicas como *surrogate models* ou *warm starting* seriam necessárias
- O espaço de hiperparâmetros considerado é discreto; refinamento contínuo (ex: CMA-ES) poderia melhorar ainda mais

In [ ]:
# Salva o sumário final em JSON para referência
sumario = {
    'baseline': {
        'fitness_medio': round(float(np.mean(fits_b)), 4),
        'fitness_max':   round(float(np.max(fits_b)), 4),
        'tempo_medio_s': round(float(np.mean(temps_b)), 4),
    },
    'meta_ga': {
        'fitness_medio': round(float(np.mean(fits_m)), 4),
        'fitness_max':   round(float(np.max(fits_m)), 4),
        'tempo_medio_s': round(float(np.mean(temps_m)), 4),
        'melhoria_pct':  round(delta, 2),
    },
    'melhor_config': {
        k: getattr(cfg_otimizada, k) for k in HIPER_KEYS
    }
}

with open(DATA_DIR / 'sumario_resultados.json', 'w') as f:
    json.dump(sumario, f, indent=2)

print('Sumário salvo em data/sumario_resultados.json')
print(json.dumps(sumario, indent=2))